# Basic Drug Screening

In [1]:
import pandas as pd
import scanpy as sc
import drugreflector as dr

# All major classes available at top level
# dr.DrugReflector, dr.SignatureRefinement, dr.compute_vscores_adata, etc.

### 第一步：加载数据

In [2]:
import scanpy as sc
import os

# 查看 scanpy 数据目录
data_dir = sc.settings.datasetdir
print("Scanpy data directory:", data_dir)

# 删除损坏的 pbmc3k_processed.h5ad 文件
corrupted_file = data_dir / "pbmc3k_processed.h5ad"
if corrupted_file.exists():
    os.remove(corrupted_file)
    print(f"Deleted corrupted file: {corrupted_file}")
else:
    print("File not found — will download fresh copy.")


Scanpy data directory: /disk1/cai026/Drugreflector/notebook/data
Deleted corrupted file: /disk1/cai026/Drugreflector/notebook/data/pbmc3k_processed.h5ad


In [3]:
# Step 1: Load PBMC 3k dataset with cell type annotations
adata = sc.datasets.pbmc3k()  # Unfiltered dataset with more genes
annots = sc.datasets.pbmc3k_processed().obs  # Cell type annotations

# Merge annotations
adata.obs = pd.merge(adata.obs, annots, how='left', left_index=True, right_index=True)

100%|████████████████████████████████████████████████████████████████████████████████████████| 23.5M/23.5M [00:04<00:00, 6.03MB/s]


In [4]:
adata

AnnData object with n_obs × n_vars = 2700 × 32738
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'gene_ids'

In [5]:
adata.obs

,n_genes,percent_mito,n_counts,louvain
index,,,,
AAACATACAACCAC-1,781.0,0.030178,2419.0,CD4 T cells
AAACATTGAGCTAC-1,1352.0,0.037936,4903.0,B cells
AAACATTGATCAGC-1,1131.0,0.008897,3147.0,CD4 T cells
AAACCGTGCTTCCG-1,960.0,0.017431,2639.0,CD14+ Monocytes
AAACCGTGTATGCG-1,522.0,0.012245,980.0,NK cells
...,...,...,...,...
TTTCGAACTCTCAT-1,1155.0,0.021104,3459.0,CD14+ Monocytes
TTTCTACTGAGGCA-1,1227.0,0.009294,3443.0,B cells
TTTCTACTTCCTCG-1,622.0,0.021971,1684.0,B cells


### 第二步：计算两个细胞群之间的差异（v-scores）

In [6]:
# Step 2: Compute v-scores between two monocyte populations
vscores = dr.compute_vscores_adata(
    adata, 
    group_col='louvain',
    group1_value='CD14+ Monocytes',    # Classical monocytes
    group2_value='FCGR3A+ Monocytes'   # Non-classical monocytes
)

print(f"V-score comparison: {vscores.name}")
print(f"Computed v-scores for {len(vscores)} genes")
print(f"Top upregulated genes in FCGR3A+ vs CD14+ monocytes:")
print(vscores.nlargest(10))

V-score comparison: louvain:CD14+ Monocytes->FCGR3A+ Monocytes
Computed v-scores for 32738 genes
Top upregulated genes in FCGR3A+ vs CD14+ monocytes:
index
RPS19     1.498847
FCGR3A    1.370333
IFITM2    1.263581
LST1      1.212525
TMSB4X    1.164625
AIF1      1.153117
FCER1G    1.114965
B2M       1.113765
MALAT1    1.069324
RHOC      1.038001
Name: louvain:CD14+ Monocytes->FCGR3A+ Monocytes, dtype: float32


### 第三步：加载预训练的模型

In [13]:
# Step 3: Initialize DrugReflector with model checkpoints
model_paths = [
    '/disk1/cai026/Drugreflector/drugreflector-main/checkpoints/model_fold_0.pt',
    '/disk1/cai026/Drugreflector/drugreflector-main/checkpoints/model_fold_1.pt', 
    '/disk1/cai026/Drugreflector/drugreflector-main/checkpoints/model_fold_2.pt'
]

In [14]:
model = dr.DrugReflector(checkpoint_paths=model_paths)

如果遇到了Weights only load failed. This file can still be loaded, to do so you have two options 的错误  
在提问区找到的答案是需要更新模型的权重文件https://github.com/Cellarity/drugreflector/issues/6#issuecomment-3452007630  
在https://zenodo.org/records/17437512  

### 第四步：预测药物

In [15]:
# Step 4: Make predictions using v-scores
# DrugReflector will automatically preprocess gene names to HGNC format
predictions = model.predict(vscores, n_top=50)
print(f"Prediction results shape: {predictions.shape}")
print(f"Columns: {predictions.columns.names}")

# Access different metrics
print("\nTop 10 predicted compounds by rank:")
rank_col = ('rank', vscores.name)  # Uses informative name: 'louvain:CD14+ Monocytes->FCGR3A+ Monocytes'
print(predictions[rank_col].nsmallest(10))

print("\nTop 10 compounds by probability:")
prob_col = ('prob', vscores.name)  
print(predictions[prob_col].nlargest(10))

print(f"\nAvailable columns: {list(predictions.columns)}")

Preprocessing gene names to HGNC format...
Preprocessed 12270/32738 gene names for HGNC compatibility
Examples of changes:
  RP11-34P13.7 -> RP11-34P13
  RP11-34P13.8 -> RP11-34P13-1
  ... and 12265 more


/disk1/cai026/mambaforge/envs/drugreflector/lib/python3.8/site-packages/drugreflector/drugreflector/ensemble_model.py:181: UserWarning: Model 0 missing 25 genes: ['ADGRE5', 'ADGRG1', 'B4GAT1', 'CARMIL1', 'CEMIP2', 'CIAO3', 'COQ8A', 'ELP1', 'ERO1A', 'HACD3']
  warnings.warn(f"Model {idx} missing {len(missing_genes)} genes: {missing_genes[:10].tolist()}")
/disk1/cai026/mambaforge/envs/drugreflector/lib/python3.8/site-packages/drugreflector/drugreflector/ensemble_model.py:181: UserWarning: Model 1 missing 25 genes: ['ADGRE5', 'ADGRG1', 'B4GAT1', 'CARMIL1', 'CEMIP2', 'CIAO3', 'COQ8A', 'ELP1', 'ERO1A', 'HACD3']
  warnings.warn(f"Model {idx} missing {len(missing_genes)} genes: {missing_genes[:10].tolist()}")
/disk1/cai026/mambaforge/envs/drugreflector/lib/python3.8/site-packages/drugreflector/drugreflector/ensemble_model.py:181: UserWarning: Model 2 missing 25 genes: ['ADGRE5', 'ADGRG1', 'B4GAT1', 'CARMIL1', 'CEMIP2', 'CIAO3', 'COQ8A', 'ELP1', 'ERO1A', 'HACD3']
  warnings.warn(f"Model {idx} 

Prediction results shape: (50, 3)
Columns: [None, None]

Top 10 predicted compounds by rank:
BRD-K03641750    0
BRD-K67578145    1
BRD-K48059230    2
BRD-K12244279    3
BRD-K57080016    4
BRD-K00662280    5
BRD-K51662849    6
BRD-K68756823    7
BRD-K99475920    8
BRD-K87909389    9
Name: (rank, louvain:CD14+ Monocytes->FCGR3A+ Monocytes), dtype: int64

Top 10 compounds by probability:
BRD-K03641750    0.032052
BRD-K67578145    0.020683
BRD-K48059230    0.012853
BRD-K12244279    0.012279
BRD-K57080016    0.009609
BRD-K00662280    0.007930
BRD-K51662849    0.007058
BRD-K68756823    0.006186
BRD-K99475920    0.005985
BRD-K87909389    0.005198
Name: (prob, louvain:CD14+ Monocytes->FCGR3A+ Monocytes), dtype: float32

Available columns: [('rank', 'louvain:CD14+ Monocytes->FCGR3A+ Monocytes'), ('logit', 'louvain:CD14+ Monocytes->FCGR3A+ Monocytes'), ('prob', 'louvain:CD14+ Monocytes->FCGR3A+ Monocytes')]


### Best Practices

- **Preferred**: Use official HGNC symbols (`TP53`, `EGFR`, `CDKN1A`)
- **Acceptable**: Mixed case, common prefixes/suffixes (automatically cleaned)
- **Check coverage**: Ensure your genes overlap with the 978 landmark genes used by the model

一定要求要有978基因重叠才行  一般都行的